## Applications of Pathwise Sensitivity Method in Simulating Paths

### Introduction

In this notebook, we consider the problem below:

*Suppose that $S$ satisfies the standard Geometric Brownian Motion:*

$$
dS_t = r S_t \, dt + \sigma S_t \, dW_t, \quad t \in [0,T]; \quad S_0 = s_0
$$

*Consider a down-and-out barrier option with the discounted payoff:*

$$
V = \mathbb{E} \left[ e^{-rT} (S_T - K)^+ \mathbf{1}_{\min_{t \in [0,T]} S_t > B} \right]
$$

*Take the parameter values:*

$$
r = 0.05, \quad \sigma = 0.5, \quad T = 1, \quad s_0 = 100, \quad K = 110, \quad B = 90
$$

*Implement the **pathwise sensitivity method** to estimate Delta and Vega for the down-and-out barrier option, incorporating the **barrier crossing technique** introduced in lectures to achieve $\mathcal{O}(h)$ weak convergence.*

Delta: $\delta = \frac{\partial V}{\partial S_0}$

Vega: $\nu = \frac{\partial V}{\partial \sigma}$ where $\sigma$ is the volatility.


### Standard Pathwise Sensitivity Method

Writing $V = V(S_T^\theta)$, we obtain the standard pathwise sensitivity formula
$$
\frac{\partial V}{\partial \theta} = \frac{\partial}{\partial \theta}\mathbb{E}\left[V(S_T^\theta)\right] = \mathbb{E}\left[\frac{\partial}{\partial \theta}V(S_T^\theta)\right] = \mathbb{E}\left[V'(S^\theta_T)\frac{\partial S^\theta_T}{\partial \theta}\right]
$$
then we can use the idea of Monte Carlo to generate $n$ paths of stock prices and estimate the graditent easily, hence calculate the sample mean as a simulation of our target. This is called the **pathwise sensitivity method**. It works perfectly if we are dealing with greeks in a *vanilla option*.

### Why the Standard Pathwise Sensitivity Method Fails

If we intend to use standard pathwise sensitivity method to simulate $\delta$, calculate:

$$
\delta = \frac{\partial V}{\partial S_0} = \frac{\partial}{\partial S_0}\mathbb{E} \left[ e^{-rT} (S_T - K)^+ \mathbf{1}_{\min_{t \in [0,T]} S_t > B} \right] = e^{-rT}\mathbb{E} \left[ \frac{\partial}{\partial S_0} (S_T - K)^+ \mathbf{1}_{\min_{t \in [0,T]} S_t > B} \right]
$$

One may spot that the indicator function $\mathbf{1}_{\min_{t \in [0,T]} S_t > B}$ is not differentiable in $S_0$, hence the expression above is undefined. Similarly, $\nu$ cannot be simulated via standard pathwise sensitivity method.

### Brownian Bridge

Let $\{W_t\}_{t \geq 0}$ be a standard Brownian motion. Define

$$
M_T = \max_{0\leq u \leq T} W_u
$$

to be the continuous-time maximum. However, while we are doing numerical simulation, we can only obtain the Euler diecrete approximation of $W$ over $[0,T]$. Let $T = nh$, define

$$
\hat{M^h}(n) = \max\{W_0,W_h,W_{2h},\dots,W_{nh}\}
$$

to be the discrete-time maximum. It is proven that
$$
h^{-\frac{1}{2}}(\hat{M^h}(n) - M_T)
$$
has a limiting distribution as $h \rightarrow 0$, which implies
$$
\hat{M^h}(n) \rightarrow M_T \;\text{  with a convergence speed of  }\; O(h^{-\frac{1}{2}})
$$

To improve, we introduce the **Brownian bridge**. This is a continuous-time gaussian process $B_t$ whose probability distribution is the conditional probability distribution of a standard Brownian motion $W_t$ subject to the condition that $W_T = w$. Therefore, both the start and end values are determined. More precisely,
$$
B_t = W_t \mid W_T = w,\; t \in \left[0,T\right]
$$





### Maximum Distribution of Brownian Bridge

We want to compute the conditional distribution

$$
\mathbb{P}(M_T \le m \mid W_T = w) = \frac{\mathbb{P}(M_T\le m,\ W_T=w)}{\mathbb{P}(W_T=w)}
$$

To compute the numerator, consider the complementary event $\mathbb{P}(M_T > m,\ W_T=w)$. By the *reflection principle*, paths that cross level $m$ and end at $w$ correspond bijectively to paths that end at $2m - w$. Hence,
$$
\mathbb{P}(M_T > m,\ W_T=w) = \mathbb{P}(W_T=2m-w)
$$
Therefore,
$$
\mathbb{P}(M_T\le m,\ W_T=w) = \mathbb{P}(W_T=w) - \mathbb{P}(W_T = 2m-w)
$$
Since $W_T \sim \mathcal{N}(0,T)$, its density is
$$
f_T(w) = \frac{1}{\sqrt{2\pi T}} e^{-w^2/(2T)}
$$
Thus,
$$
\frac{\mathbb{P}(W_T=2m-w)}{\mathbb{P}(W_T=w)} = \frac{ e^{-(2m-w)^2/(2T)}}{e^{-w^2/(2T)}} =
\exp\left(-\frac{2m(m-w)}{T}\right)
$$
and hence
$$
\mathbb{P}(M_T \le m \mid W_T = w) = \frac{\mathbb{P}(W_T=w) - \mathbb{P}(W_T = 2m-w)}{\mathbb{P}(W_T=w) } = 1 - \exp\left(-\frac{2m(m-w)}{T}\right) =: F_w(m) \;\;\; (1)
$$

### Sampling $M_T$

Let $U \sim \mathcal{U}\left[0,1\right]$, then let $M = F_w^{-1}(U)$, we have
$$
\mathbb{P}(M\leq m) = \mathbb{P}(F_w^{-1}(U) \leq m) = \mathbb{P}(U \leq F_w(m)) = F_w(m)
$$
so $M$ is a random variable in the distribution with CDF $F_w$.

Let's find a expression of $M$:
\begin{align}
    &U = F_w(M) = 1 - \exp{\left(-\frac{2M(M - w)}{T}\right)}\\
    \Rightarrow \;\;\; &-\frac{2M(M - w)}{T} = \log(1 - U)\\
    \Rightarrow \;\;\; &M^2 - wM + \frac{T}{2}\log U = 0 \quad \text{since}\; U \triangleq 1 - U \\
    \Rightarrow \;\;\; &M = \frac{w \pm \sqrt{w^2 - 2T\log U}}{2}\\
    \Rightarrow \;\;\; &M = \frac{w + \sqrt{w^2 - 2T\log U}}{2} \quad \text{since we expect}\; M \geq w \;\text{almost surely}
\end{align}

Therefore, we find out a way to sample $M_T$ directly as follows:

- Generate $W_T \sim \mathcal{N}(0,T)$;
- Conditional on $W_T$, the process $\{W_t\}_{0\leq t \leq T}$ becomes a **Brownian bridge**;
- Given $W_T$, let $U \sim \mathcal{U}\left[0,1\right]$, define
$$
M_T = \frac{w + \sqrt{w^2 - 2T\log U}}{2}
$$
- Generate $M_T$ by generating samples $U$.

In the discrete setting, suppose we have had the end points $W_{ih}$ and $W_{(i+1)h}$, then the maximum of the interpolating Brownian bridge $\{\hat{M_i}\}_{0 \leq i \leq n-1}$ is
$$
\hat{M_i} = \frac{W_{ih} + W_{(i+1)h} + \sqrt{\left(W_{(i+1)h} - W_{ih}\right)^2 - 2b_i^2h\log U_i}}{2}.
$$
Hence $M_T$ can be simulated as
$$
\hat{M_T} = \max \{M_0, M_1, \dots,M_{n-1}\}
$$

### Barrier Crossing Method

Let's go back to our problem and see how this idea is going to solve the problem of discontinuity.

In the down-and-out barrier option, we have a barrier $B$ and the initial stock price $S_0 > B$. The payoff (without the discounting factor) is
$$
V = (S_T - K)^+ \mathbf{1}_{\min_{t \in [0,T]} S_t > B}
$$
Let $t_i = ih$ for $0\leq i \leq n$, where $n = \frac{T}{h}$. For each interval $\left[t_i, t_{i+1}\right]$, instead of considering whether the two end points $S_{t_i},S_{t_{i+1}}$ are below $B$, we now consider $\mathbb{P}(S_t \geq B \;\forall t \in \left[t_i, t_{i+1}\right] \mid S_{t_i},S_{t_{i+1}})$, namely the probability that the stock price stays above $B$ for the entire interval.

Since $S_t$ is a geometric Brownian motion, it is easier to consider $X_t := \log S_t$. By *Ito's formula*, we have
$$
dX_t = \left(r - \frac{1}{2}\sigma^2\right)dt + \sigma dW_t
$$
and given $X_{t_i}, X_{t_{i+1}}$, the path in the middle is a Brownian bridge. Therefore, we need to consider the probability that the Brownian bridge stays above $\log B$ in $\left[t_i, t_{i+1}\right]$. We call this **one-step survival probability**. Calculate:
\begin{align}
    &\mathbb{P}\left(\min_{t \in \left[t_i, t_{i+1}\right]} S_t > B \mid S_{t_i} = s_i, S_{t_{i+1}} = s_{i+1}\right)\\
    = &\mathbb{P}\left(\min_{t \in \left[t_i, t_{i+1}\right]} X_t > b \mid X_{t_i} = x_i, X_{t_{i+1}} = x_{i+1}\right) \quad \text{where }b = \log B, x_i = \log s_i \text{ and }x_{i+1} = \log s_{i+1}\\
    = &\mathbb{P}\left(\min_{t \in \left[t_i, t_{i+1}\right]} Y_t > b - x_i \mid Y_{t_i} = 0, Y_{t_{i+1}}= x_{i+1} - x_i\right) \quad \text{where }Y_t = X_t - x_i \\
    = &\mathbb{P}\left(\max_{t \in \left[t_i, t_{i+1}\right]} Z_t < x_i - b \mid Z_{t_i} = 0, Z_{t_{i+1}}= x_{i} - x_{i+1}\right) \quad \text{where }Z_t = -Y_t \\
    = &\mathbb{P}\left(\max_{t \in \left[t_i, t_{i+1}\right]} W_t < \frac{1}{\sigma}(x_i - b) \mid W_{t_i} = 0, W_{t_{i+1}}= \frac{1}{\sigma}(x_{i} - x_{i+1})\right) \quad \text{where }W_t = \frac{1}{\sigma}Z_t\\
    = &1 - \exp \left(-\frac{2(x_i - b)(x_{i+1} - b)}{\sigma^2 h}\right) \quad \text{by (1)}
\end{align}
Notice that the transformation $W_t = \frac{1}{\sigma}Z_t$ is essential because it leads to the diffusion coefficient to be 1, so that we can apply (1) which is based on standard Brownian motion.

Denote $p_i$ to be the one-step survival probability in $\left[t_i, t_{i+1}\right]$, then
$$
p_i =
\begin{cases}
1 - \exp\left(-\frac{2(\log(S_i / B))(\log(S_{i+1} / B))}{\sigma^2 h}\right), & S_i > B,\; S_{i+1} > B, \\
0, & \text{otherwise}.
\end{cases}
$$
Given the entire discrete path $S_{t_0}, S_{t_1}, \dots, S_{t_n}$, the conditional probability such that the entire continuous path is above the barrier $B$ can be approximated by
$$
P = \prod_{i = 0}^{n - 1}p_i
$$
Replacing the indicator $\mathbf{1}_{\min_{t \in [0,T]} S_t > B}$ by $P$, we finally obtain a new version of payoff function
$$
\tilde{V} = (S_T - K)^+\prod_{i = 0}^{n - 1}p_i
$$
and this is now differentiable in both $s_0$ and $\sigma$.

This method is proved to have a convergence speed of $O(h)$.

### Algorithm

1. Generate the discrete GBM paths for stock price $S_t$ by
$$
S_{t_{i+1}} = S_{t_i} \exp \left[\left(r - \frac{1}{2}\sigma^2\right)h + \sigma \sqrt{h} Z_i\right]
\quad Z_i \sim \mathcal{N}(0,1)
$$
2. Calculate the one-step survival probability $p_i$ for each interval;
3. Use $\tilde{V}$ as the price estimator;
4. Apply pathwise differentiation on the new payoff function with respect to $s_0$ and $\sigma$;
5. Repeat steps 1 - 4 and calculate the sample mean of $\delta$ and $\nu$.

In [19]:
import numpy as np

r_default = 0.05
sigma_default = 0.5
T_default = 1.0
s0_default = 100.0
K_default = 110.0
B_default = 90.0

In [29]:
"""
Monte Carlo estimator for a down-and-out call under GBM, using pathwise sensitivities and Brownian-bridge barrier correction.
"""
def estimate_down_and_out_call_pathwise(
    n_paths=1000000,
    n_steps=100,
    batch_size=10000,
    seed=42,
    r = r_default, sigma = sigma_default, T = T_default, s0 = s0_default, K = K_default, B = B_default):

    h = T / n_steps                         # gap between each step
    sqrt_h = np.sqrt(h)                     # used to simulate paths
    disc = np.exp(-r * T)                   # discounting factor
    rng = np.random.default_rng(seed)       # Random number generator by the preset seed

    # Running totals for sample means (targets) and second moments (to calculate standard errors)
    sum_price, sum_price_sq = 0., 0.
    sum_delta, sum_delta_sq = 0., 0.
    sum_vega , sum_vega_sq  = 0., 0.

    n_done = 0

    while n_done < n_paths:
        m = min(batch_size, n_paths - n_done)

        # State variables along each path
        S = np.full(m, s0, dtype=np.float64)

        # Pathwise sensitivities of S wrt s0 and sigma
        dS_ds0 = np.ones(m, dtype=np.float64)      # dS_0/ds0 = 1
        dS_dsigma = np.zeros(m, dtype=np.float64)  # dS_0/dsigma = 0

        # Survival probability product P = prod_i p_i
        P_surv = np.ones(m, dtype=np.float64)

        # Pathwise sensitivities of P_surv
        dP_ds0 = np.zeros(m, dtype=np.float64)
        dP_dsigma = np.zeros(m, dtype=np.float64)

        for _ in range(n_steps):
            Z = rng.standard_normal(m)

            drift = (r - 0.5 * sigma**2) * h
            diffusion = sigma * sqrt_h * Z
            G = np.exp(drift + diffusion)

            S_next = S * G

            # Pathwise derivatives of the exact GBM step
            # S_{i+1} = S_i * exp((r - 0.5 sigma^2) h + sigma sqrt(h) Z_i)
            dS_ds0_next = dS_ds0 * G

            a = -sigma * h + sqrt_h * Z
            dS_dsigma_next = G * (dS_dsigma + S * a)

            # One-step Brownian-bridge survival probability in log-space
            # p_i = 1 - exp( - 2 log(S_i/B) log(S_{i+1}/B) / (sigma^2 h) )
            # if both endpoints are above the barrier; otherwise p_i = 0
            p_i = np.zeros(m, dtype=np.float64)
            dp_ds0_i = np.zeros(m, dtype=np.float64)
            dp_dsigma_i = np.zeros(m, dtype=np.float64)

            mask = (S > B) & (S_next > B)

            if np.any(mask):
                S_m = S[mask]
                S_next_m = S_next[mask]

                dS_ds0_m = dS_ds0[mask]
                dS_ds0_next_m = dS_ds0_next[mask]

                dS_dsigma_m = dS_dsigma[mask]
                dS_dsigma_next_m = dS_dsigma_next[mask]

                L0 = np.log(S_m / B)
                L1 = np.log(S_next_m / B)

                A = 2.0 * L0 * L1 / (sigma**2 * h)
                exp_minus_A = np.exp(-A)

                p_loc = 1.0 - exp_minus_A
                p_i[mask] = p_loc

                # d/dtheta log(S/B) = (1/S) dS/dtheta
                dL0_ds0 = dS_ds0_m / S_m
                dL1_ds0 = dS_ds0_next_m / S_next_m

                dL0_dsigma = dS_dsigma_m / S_m
                dL1_dsigma = dS_dsigma_next_m / S_next_m

                # A = 2 L0 L1 / (sigma^2 h)
                dA_ds0 = (2.0 / (sigma**2 * h)) * (dL0_ds0 * L1 + L0 * dL1_ds0)

                dA_dsigma = (
                    (2.0 / (sigma**2 * h)) * (dL0_dsigma * L1 + L0 * dL1_dsigma)
                    - (4.0 * L0 * L1) / (sigma**3 * h)
                )

                # p = 1 - exp(-A)  => dp = exp(-A) dA
                dp_ds0_i[mask] = exp_minus_A * dA_ds0
                dp_dsigma_i[mask] = exp_minus_A * dA_dsigma

            # Recursive product rule for P_surv
            old_P = P_surv.copy()

            dP_ds0 = dP_ds0 * p_i + old_P * dp_ds0_i
            dP_dsigma = dP_dsigma * p_i + old_P * dp_dsigma_i
            P_surv = old_P * p_i

            # Move to next step
            S = S_next
            dS_ds0 = dS_ds0_next
            dS_dsigma = dS_dsigma_next

        # Terminal call payoff and its pathwise derivatives
        call = np.maximum(S - K, 0.0)
        itm = (S > K).astype(np.float64)

        dcall_ds0 = itm * dS_ds0
        dcall_dsigma = itm * dS_dsigma

        # Discounted corrected payoff
        price_paths = disc * call * P_surv

        # Pathwise Greeks
        delta_paths = disc * (dcall_ds0 * P_surv + call * dP_ds0)
        vega_paths = disc * (dcall_dsigma * P_surv + call * dP_dsigma)

        # Accumulate statistics for fair prices, delta and vega
        sum_price += np.sum(price_paths)
        sum_price_sq += np.sum(price_paths**2)
        sum_delta += np.sum(delta_paths)
        sum_delta_sq += np.sum(delta_paths**2)
        sum_vega += np.sum(vega_paths)
        sum_vega_sq += np.sum(vega_paths**2)

        n_done += m

    # Sample means
    price = sum_price / n_paths
    delta = sum_delta / n_paths
    vega = sum_vega / n_paths

    # Unbiased sample variances / MC standard errors
    def mc_se(sum_x, sum_x_sq, n):
        mean_x = sum_x / n
        var_x = (sum_x_sq - n * mean_x**2) / (n - 1)
        return np.sqrt(var_x / n)

    price_se = mc_se(sum_price, sum_price_sq, n_paths)
    delta_se = mc_se(sum_delta, sum_delta_sq, n_paths)
    vega_se = mc_se(sum_vega, sum_vega_sq, n_paths)

    return {
        "parameters": {
            "r": r,
            "sigma": sigma,
            "T": T,
            "s0": s0,
            "K": K,
            "B": B,
            "n_paths": n_paths,
            "n_steps": n_steps,
            "h": h,
        },
        "price": price,
        "price_se": price_se,
        "delta": delta,
        "delta_se": delta_se,
        "vega": vega,
        "vega_se": vega_se,
    }


In [30]:
results = estimate_down_and_out_call_pathwise()

print("Parameters:")
for k, v in results["parameters"].items():
    print(f"  {k} = {v}")

print("\nMonte Carlo estimates:")
print(f"  Price = {results['price']:.4f}   (Standard Monte Carlo Error = {results['price_se']:.4f})")
print(f"  Delta = {results['delta']:.4f}   (Standard Monte Carlo Error = {results['delta_se']:.4f})")
print(f"  Vega  = {results['vega']:.4f}   (Standard Monte Carlo Error = {results['vega_se']:.4f})")

Parameters:
  r = 0.05
  sigma = 0.5
  T = 1.0
  s0 = 100.0
  K = 110.0
  B = 90.0
  n_paths = 1000000
  n_steps = 100
  h = 0.01

Monte Carlo estimates:
  Price = 8.6169   (Standard Monte Carlo Error = 0.0301)
  Delta = 0.8519   (Standard Monte Carlo Error = 0.0040)
  Vega  = 4.5587   (Standard Monte Carlo Error = 0.1136)
